# VLM image-pair evaluation

Evaluate image pairs created by `bonsai/comfy-gen-lora`. This notebook is deliberately separate from generation. It validates the dataset, evaluates pending pairs, and stores resumable JSONL results.

**Model adapter:** implement either `evaluate_pair_local()` or `evaluate_pair_cloud()` below. Keep credentials in Kaggle Secrets, never in this notebook.

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

# Update this to the mounted Kaggle Dataset path.
DATASET_ROOT = Path('/kaggle/input/lora-vlm-evaluation-data')
MANIFEST_PATH = DATASET_ROOT / 'manifest.jsonl'
RESULTS_DIR = Path('/kaggle/working/evaluations')
RESULTS_PATH = RESULTS_DIR / 'results.jsonl'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert MANIFEST_PATH.is_file(), f'Missing manifest: {MANIFEST_PATH}'
print('Dataset:', DATASET_ROOT)
print('Manifest:', MANIFEST_PATH)
print('Results:', RESULTS_PATH)

In [ ]:
def load_manifest(path):
    rows = []
    with path.open(encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            if line.strip():
                row = json.loads(line)
                if not row.get('experiment_id'):
                    raise ValueError(f'Line {line_no}: experiment_id is required')
                for key in ('lora_image', 'baseline_image'):
                    if not row.get(key):
                        raise ValueError(f'Line {line_no}: {key} is required')
                rows.append(row)
    ids = [r['experiment_id'] for r in rows]
    if len(ids) != len(set(ids)):
        raise ValueError('Duplicate experiment_id values in manifest')
    return rows

pairs = load_manifest(MANIFEST_PATH)
print(f'Loaded {len(pairs)} image pairs')
for pair in pairs[:5]:
    print(pair['experiment_id'], pair['lora_image'], '<->', pair['baseline_image'])

In [ ]:
def image_path(relative_path):
    path = (DATASET_ROOT / relative_path).resolve()
    root = DATASET_ROOT.resolve()
    if root not in path.parents and path != root:
        raise ValueError(f'Path escapes dataset root: {relative_path}')
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

for pair in pairs:
    image_path(pair['lora_image'])
    image_path(pair['baseline_image'])
print('All referenced image files exist.')

## VLM adapter

Implement **one** adapter for your chosen vision model. The function must return a JSON-serializable dictionary. A cloud adapter should read its API key from `kaggle_secrets.UserSecretsClient`; a local adapter should load a vision-capable model inside the Kaggle runtime. `localhost` in Kaggle refers to the Kaggle VM, not your personal PC.

In [ ]:
EVALUATION_PROMPT = '''Compare image A (trained LoRA) and image B (same base model without LoRA).
Return JSON only. Evaluate visible evidence, not assumptions.
For each image classify woman, snake, and PC (monitor/keyboard) as present, partial, absent, or uncertain.
State whether all three are visible together. Describe only observable style/quality differences.
Do not claim that one pair proves causation. Include uncertainties.
Schema: {"subjects":{"woman":{"A":"...","B":"..."},"snake":{"A":"...","B":"..."},"pc":{"A":"...","B":"..."}},"all_three_visible_together":{"A":false,"B":false},"style_differences":[],"image_quality":{"A":"","B":""},"lora_effect_evidence":[],"uncertainties":[]}
'''

def evaluate_pair_local(image_a_path, image_b_path):
    # TODO: connect a local vision model loaded in this Kaggle runtime.
    raise NotImplementedError('Choose and configure a local vision model adapter first.')

def evaluate_pair_cloud(image_a_path, image_b_path):
    # TODO: connect your chosen cloud vision API using Kaggle Secrets.
    raise NotImplementedError('Choose and configure a cloud vision API adapter first.')

# Choose after implementing the corresponding adapter:
EVALUATOR = evaluate_pair_local  # or evaluate_pair_cloud

In [ ]:
def completed_ids(path):
    done = set()
    if path.exists():
        with path.open(encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        row = json.loads(line)
                        if row.get('status') == 'success':
                            done.add(row['experiment_id'])
                    except (json.JSONDecodeError, KeyError):
                        continue
    return done

done = completed_ids(RESULTS_PATH)
pending = [p for p in pairs if p['experiment_id'] not in done]
print(f'Completed: {len(done)} | Pending: {len(pending)}')

for pair in pending:
    exp_id = pair['experiment_id']
    record = {
        'experiment_id': exp_id,
        'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
        'evaluator': getattr(EVALUATOR, '__name__', str(EVALUATOR)),
        'status': 'success',
    }
    try:
        result = EVALUATOR(image_path(pair['lora_image']), image_path(pair['baseline_image']))
        record['result'] = result
    except Exception as exc:
        record['status'] = 'error'
        record['error'] = f'{type(exc).__name__}: {exc}'
    with RESULTS_PATH.open('a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\\n')
    print(exp_id, record['status'])

print('Results saved to:', RESULTS_PATH)

## Output

The result file is `/kaggle/working/evaluations/results.jsonl`. Download it from the Kaggle output or publish it as a new Dataset version. Successful experiment IDs are skipped on rerun; error records are retried.